# 🚀 Tech Stack Recommender
### DecodeLabs — Project 3 Capstone
**Pipeline:** Ingestion → TF-IDF Scoring → Cosine Similarity → Top-N Filter

## Step 1: Ingestion — Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv("raw_skills.csv")

print(df.head())

jobs = df.to_dict(orient="records")
for job in jobs:
    print(job["job_role"])
    print(job["skills"])

                    job_role  \
0             Data Scientist   
1  Machine Learning Engineer   
2               Data Analyst   
3              Data Engineer   
4                AI Engineer   

                                              skills  
0  Python,SQL,Machine Learning,Statistics,Data An...  
1  Python,Machine Learning,Deep Learning,TensorFl...  
2  SQL,Excel,Data Analysis,Data Visualization,Pyt...  
3  Python,SQL,Spark,Hadoop,ETL,Airflow,Data Pipel...  
4  Python,Deep Learning,TensorFlow,PyTorch,NLP,Co...  
Data Scientist
Python,SQL,Machine Learning,Statistics,Data Analysis,Pandas,NumPy,Scikit-learn,Data Visualization,Deep Learning
Machine Learning Engineer
Python,Machine Learning,Deep Learning,TensorFlow,PyTorch,Scikit-learn,MLOps,Docker,Statistics,NumPy
Data Analyst
SQL,Excel,Data Analysis,Data Visualization,Python,Tableau,Power BI,Statistics,Reporting,Business Intelligence
Data Engineer
Python,SQL,Spark,Hadoop,ETL,Airflow,Data Pipelines,Cloud,AWS,Docker
AI Engineer
Python,

## Step 2: Get User Skills Input

In [2]:
print("Enter at least 3 skills (separated by commas OR spaces)")
print("Example: python, sql, machine learning")

user = input("> ").lower()

# Accept both commas and spaces as separators
if "," in user:
    user_skills = [s.strip() for s in user.split(",")]
else:
    user_skills = [s.strip() for s in user.split()]

# remove empty strings
user_skills = [s for s in user_skills if s]

if len(user_skills) < 3:
    print("Please enter at least 3 skills.")
else:
    print("Your skills:", user_skills)

Enter at least 3 skills (separated by commas OR spaces)
Example: python, sql, machine learning
Your skills: ['java', 'sql', 'html']


## Step 3: Build Shared Vocabulary

In [3]:
vocab = set()

for job in jobs:
    # split by comma since skills are stored as "Python,SQL,..."
    words = [w.strip().lower() for w in job["skills"].split(",")]

    for w in words:
        vocab.add(w)

for w in user_skills:
    vocab.add(w)

vocab = list(vocab)
print(f"Vocabulary size: {len(vocab)} unique skills")

Vocabulary size: 111 unique skills


## Step 4: TF — Term Frequency

In [4]:
def tf(document):
    # split by comma for job docs, already a list for user
    if isinstance(document, str):
        words = [w.strip().lower() for w in document.split(",")]
    else:
        words = document

    tf_dict = {}

    for word in vocab:

        count = words.count(word)

        tf_dict[word] = count / len(words)

    return tf_dict

## Step 5: IDF — Inverse Document Frequency

In [5]:
df = {}

N = len(jobs) + 1

for word in vocab:

    count = 0

    for job in jobs:
        job_words = [w.strip().lower() for w in job["skills"].split(",")]

        if word in job_words:
            count += 1

    if word in user_skills:
        count += 1

    df[word] = count

import math

idf = {}

for word in vocab:

    idf[word] = math.log(N / df[word])

## Step 6: TF-IDF Vectorization

In [6]:
def tfidf(document):

    tf_values = tf(document)

    vector = []

    for word in vocab:

        vector.append(tf_values[word] * idf[word])

    return vector

job_vectors = []

for job in jobs:

    job_vectors.append({
        "job": job["job_role"],
        "vector": tfidf(job["skills"])
    })

user_vector = tfidf(user_skills)

## Step 7: Cosine Similarity

In [7]:
def cosine(v1, v2):

    dot = 0

    for i in range(len(v1)):
        dot += v1[i] * v2[i]

    mag1 = 0
    mag2 = 0

    for x in v1:
        mag1 += x*x

    for x in v2:
        mag2 += x*x

    mag1 = math.sqrt(mag1)
    mag2 = math.sqrt(mag2)

    if mag1 == 0 or mag2 == 0:
        return 0

    return dot / (mag1 * mag2)

scores = []

for job in job_vectors:

    score = cosine(user_vector, job["vector"])

    scores.append((job["job"], score))

## Step 8: Sort & Filter — Top 3 Results

In [8]:
scores.sort(key=lambda x: x[1], reverse=True)

print("\nTop Recommended Jobs\n")

for job, score in scores[:3]:

    print(job, "->", round(score, 3))


Top Recommended Jobs

Full Stack Developer -> 0.307
Backend Developer -> 0.236
Software Engineer -> 0.205
